# Automatic Differentiation for Least Squares

This notebook demonstrates how to compute the gradient and Hessian of a least squares objective using four different frameworks: NumPy (manual derivation), Autograd, PyTorch, and JAX. By the end, you will understand how each framework implements automatic differentiation (AD) and be able to choose the right tool for your optimization workflows.

## Problem Setup

Given the least squares function:

$$
f(\mathbf{x}) = \frac{1}{2} \|\mathbf{A} \mathbf{x} - \mathbf{b}\|^2 = \frac{1}{2} (\mathbf{A} \mathbf{x} - \mathbf{b})^\top (\mathbf{A} \mathbf{x} - \mathbf{b})
$$

where $\mathbf{A} \in \mathbb{R}^{m \times n}$ is a matrix, $\mathbf{x} \in \mathbb{R}^n$ is a vector of variables, and $\mathbf{b} \in \mathbb{R}^m$ is a vector of constants, we can derive closed-form expressions for the gradient and Hessian.

**Gradient:**
$$
\nabla f(\mathbf{x}) = \mathbf{A}^\top (\mathbf{A} \mathbf{x} - \mathbf{b})
$$

**Hessian:**
$$
\nabla^2 f(\mathbf{x}) = \mathbf{A}^\top \mathbf{A}
$$

Note that the Hessian is constant (independent of $\mathbf{x}$), which is a hallmark of quadratic objectives.

## 1. Manual Computation with NumPy

We start by implementing the gradient and Hessian directly from the closed-form expressions above. This serves as our ground truth: every AD framework should reproduce these values (up to floating-point precision). We generate a random $4 \times 3$ matrix $\mathbf{A}$ and a vector $\mathbf{b} \in \mathbb{R}^4$, then evaluate both quantities at a random point $\mathbf{x} \in \mathbb{R}^3$.

In [ ]:
import numpy as np

# Generate random A and b
np.random.seed(42)  # For reproducibility
A = np.random.rand(4, 3)  # A 4x3 matrix
b = np.random.rand(4)  # A vector of size 4

# Define functions for gradient and Hessian
def least_squares_gradient_hessian(A, b, x):
    # Gradient: A^T (A x - b)
    gradient = A.T @ (A @ x - b)

    # Hessian: A^T A
    hessian = A.T @ A

    return gradient, hessian

# Test with a random x vector
x = np.random.rand(3)  # A vector of size 3

# Compute gradient and hessian
gradient, hessian = least_squares_gradient_hessian(A, b, x)
gradient, hessian

## 2. Autograd

[Autograd](https://github.com/hips/autograd) is a lightweight Python library that can automatically differentiate native NumPy code. It works by **tracing** the computation: when you call a function, Autograd records every elementary operation on a computational tape and then walks the tape backward to accumulate gradients via the chain rule. This is **reverse-mode AD**, which is efficient when the number of outputs is small relative to the number of inputs (the typical setting for scalar loss functions).

Autograd provides `jacobian` and `hessian` as convenience wrappers. The Hessian can equivalently be computed as the Jacobian of the gradient, i.e., `jacobian(jacobian(f))`, which we verify below.

In [ ]:
import autograd.numpy as np
from autograd import jacobian, hessian

# Define least squares function (autograd requires the differentiation
# variable to be the first argument)
def least_squares(x, A, b):
    return 0.5 * np.sum((A @ x - b) ** 2)

# Gradient via reverse-mode AD (jacobian of a scalar function = gradient)
gradient_autograd = jacobian(least_squares)(x, A, b)

# Hessian via the built-in hessian() wrapper
hessian_autograd1 = hessian(least_squares)(x, A, b)

# Equivalently, Hessian = Jacobian of the gradient
hessian_autograd2 = jacobian(jacobian(least_squares))(x, A, b)

print("Gradient (autograd):", gradient_autograd)
print("Hessian (autograd):", hessian_autograd1)
print("Hessian (autograd, nested jacobian):", hessian_autograd2)

## 3. PyTorch

[PyTorch](https://pytorch.org/) uses a **dynamic computational graph** (also called define-by-run). Each time you execute operations on tensors with `requires_grad=True`, PyTorch builds a directed acyclic graph on the fly. Calling `loss.backward()` traverses this graph in reverse to compute gradients -- this is **reverse-mode AD** implemented via tape-based backpropagation.

For second-order derivatives (the Hessian), PyTorch provides `torch.autograd.functional.hessian`, which internally performs repeated backward passes. Note that PyTorch uses 32-bit floats by default, so expect small numerical differences compared to the 64-bit NumPy baseline.

In [ ]:
import torch

# Define least squares function using PyTorch operations
def least_squares_torch(x, A, b):
    return 0.5 * torch.sum((A @ x - b) ** 2)

# Convert numpy arrays to PyTorch tensors (float32 by default)
A_torch = torch.tensor(A, dtype=torch.float32)
b_torch = torch.tensor(b, dtype=torch.float32)
x_torch = torch.tensor(x, dtype=torch.float32, requires_grad=True)  # track gradients

# Forward pass: compute scalar loss
loss = least_squares_torch(x_torch, A_torch, b_torch)

# Backward pass: populate x_torch.grad with df/dx
loss.backward()
gradient_torch = x_torch.grad

# Hessian via torch.autograd.functional.hessian (uses repeated backward passes)
hessian_torch = torch.autograd.functional.hessian(
    lambda x: least_squares_torch(x, A_torch, b_torch), x_torch
)

print("Gradient (PyTorch):", gradient_torch)
print("Hessian (PyTorch):", hessian_torch)

## 4. JAX

[JAX](https://github.com/google/jax) takes a **functional transformation** approach to AD. Rather than recording a tape at runtime, JAX traces your function once to build an intermediate representation (using XLA) and then transforms it. The key primitives are:

- `jax.grad` -- reverse-mode AD, returns a function that computes the gradient.
- `jax.jacfwd` -- forward-mode AD, computes Jacobians by propagating tangent vectors forward through the computation.
- `jax.jacrev` -- reverse-mode AD for Jacobians.

To compute the Hessian, we compose `jax.jacfwd(jax.grad(f))`: the inner `grad` gives us the gradient via reverse mode, and the outer `jacfwd` differentiates the gradient via forward mode. This forward-over-reverse composition is generally the most efficient way to compute full Hessians.

In [ ]:
import jax
import jax.numpy as jnp

# Define least squares function using JAX numpy
def least_squares_jax(x, A, b):
    return 0.5 * jnp.sum((A @ x - b) ** 2)

# Convert numpy arrays to JAX arrays
A_jax = jnp.array(A)
b_jax = jnp.array(b)
x_jax = jnp.array(x)

# Gradient via reverse-mode AD: jax.grad returns a *function*
gradient_jax = jax.grad(least_squares_jax)(x_jax, A_jax, b_jax)

# Hessian via forward-over-reverse: jacfwd differentiates the gradient function
hessian_jax = jax.jacfwd(jax.grad(least_squares_jax))(x_jax, A_jax, b_jax)

print("Gradient (JAX):", gradient_jax)
print("Hessian (JAX):", hessian_jax)

## Framework Comparison

| Feature | NumPy (manual) | Autograd | PyTorch | JAX |
|---|---|---|---|---|
| **AD mechanism** | None (closed-form) | Tape-based reverse mode | Dynamic graph reverse mode | Functional transformations (forward + reverse) |
| **Hessian support** | Manual derivation | `hessian()` or nested `jacobian` | `torch.autograd.functional.hessian` | `jax.jacfwd(jax.grad(f))` |
| **GPU acceleration** | No | No | Yes | Yes |
| **Default precision** | float64 | float64 | float32 | float32 |
| **Ecosystem** | General scientific computing | Drop-in NumPy replacement | Deep learning, production deployment | Research, XLA compilation, `vmap`/`pmap` |
| **Best suited for** | Small problems with known derivatives | Prototyping, education | Training neural networks, production | High-performance research, composable transformations |

**When to use which:**

- **NumPy**: When you have a simple function with a known closed-form gradient and do not need AD machinery.
- **Autograd**: Quick prototyping when you want AD on existing NumPy code with minimal changes. Not actively maintained; consider JAX as its successor.
- **PyTorch**: The standard choice for deep learning. Its dynamic graph is convenient for models with control flow (e.g., variable-length sequences). Rich ecosystem of optimizers, data loaders, and pre-trained models.
- **JAX**: Ideal for research requiring composable transformations (`grad`, `vmap`, `jit`, `pmap`). Excellent for batched Hessian computations and large-scale scientific computing on accelerators.

## Summary

**Key takeaways from this notebook:**

1. **Automatic differentiation is not finite differences.** AD computes exact derivatives (up to floating-point precision) by systematically applying the chain rule to elementary operations. All four frameworks reproduced the same gradient and Hessian values.

2. **Reverse mode vs. forward mode.** Reverse-mode AD (used by Autograd, PyTorch, and `jax.grad`) is efficient for functions $f: \mathbb{R}^n \to \mathbb{R}$ because it computes the full gradient in a single backward pass. Forward-mode AD (`jax.jacfwd`) propagates one directional derivative at a time and is more efficient for functions $f: \mathbb{R} \to \mathbb{R}^m$. For the Hessian, composing forward-over-reverse is generally optimal.

3. **Framework choice depends on context.** For production deep learning, PyTorch is the standard. For research requiring composable transformations and hardware acceleration, JAX excels. For quick prototyping on NumPy code, Autograd (or its successor JAX) is convenient.